In [ ]:
# Sprint 51 – Task 2
# Automated Mandarin Inference, Evaluation, and Rollback Pipeline

# Cell 1 — Install dependencies
!pip install -q unsloth transformers datasets peft accelerate bitsandbytes sentencepiece langdetect sacrebleu

In [ ]:
# Cell 2 — Imports

import os
import gc
import re
import json
import math
import time
import random
import shutil
import hashlib
import statistics
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient
from typing import List, Dict, Optional, Any, Callable

import numpy as np
import pandas as pd
import torch

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import cohen_kappa_score
from langdetect import DetectorFactory

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from peft import PeftModel
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("kaggle-glm-eval")

DetectorFactory.seed = 42

print("✓ Imports ready")

In [ ]:
# Cell 3 — Config
RUN_ID = f"sprint51_task2_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

CONFIG = {
    "run_id": RUN_ID,
    "dry_run": False,
    "dry_run_n": 8,
    "seed": 3407,

    "candidate": {
        "base_model": "unsloth/Llama-3.2-1B-bnb-4bit",
        "adapter_path": "/kaggle/input/notebooks/abdighaz/sprint51-task1-full-finetuning-pipeline/task1_en2zh_outputs/final",
        "merged_model_path": "/kaggle/input/notebooks/abdighaz/sprint51-task1-full-finetuning-pipeline/task1_en2zh_outputs/final_merged_16bit",
        "candidate_manifest_path": "/kaggle/input/notebooks/abdighaz/sprint51-task1-full-finetuning-pipeline/task1_en2zh_outputs/manifests/candidate_manifest.json",
        "candidate_type": "lora_adapter",   # lora_adapter | merged_model
        "version": "candidate-v1",
        "language_direction": "en->zh",
    },

    "baseline": {
        "mode": "reconstruct",              # reconstruct | fixture
        "version": "baseline-reconstructed-v1",
        "reconstruct_base_model": "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
        "fixture_path": "/kaggle/input/task2-fixtures/baseline_fixture.jsonl",
    },

    "data": {
        "data_manifest_path": "/kaggle/input/notebooks/abdighaz/sprint51-task1-prepare-mandarin-data/mandarin_task1_dataset/manifests/data_manifest.json",
        "prompt_validation_path": "/kaggle/input/notebooks/abdighaz/sprint51-task1-prepare-mandarin-data/mandarin_task1_dataset/data/processed/prompt_validation.jsonl",
        "finetune_train_path": "/kaggle/input/notebooks/abdighaz/sprint51-task1-prepare-mandarin-data/mandarin_task1_dataset/data/processed/finetune_train.jsonl",
        "sealed_final_test_path": "/kaggle/input/notebooks/abdighaz/sprint51-task1-prepare-mandarin-data/mandarin_task1_dataset/data/sealed/final_test.jsonl",
        "gold_set_csv_path": "/kaggle/input/datasets/abdighaz/gold-set/gold_set_labeled_chatgpt.csv",
        "eval_sample_size": 200,
    },

    "judge": {
        "primary_model": "Qwen/Qwen3-8B",
        "fallback_model": "Qwen/Qwen3-4B-Instruct-2507",
        "max_seq_length": 4096,
        "max_new_tokens": 256,
        "temperature": 0.0,
        "load_in_4bit": True,
        "num_runs": 3,
        "calibration_thresholds": {
            "min_coverage": 0.90,
            "min_spearman_rho": 0.45,
            "min_mean_confidence": 0.70,
        }
    },

    "policy": {
        "policy_version": "task2-readiness-policy-v1",
        "advance_thresholds": {
            "min_quality_score": 8.0,
            "min_delta_vs_baseline": 0.0,
            "max_wrong_language_rate": 0.02,
            "max_invalid_output_rate": 0.02,
            "max_safety_violation_rate": 0.00,
            "max_memorization_flag_rate": 0.00,
            "max_format_invalid_rate": 0.02,
            "max_failure_rate": 0.02,
            "max_latency_p95_seconds": 8.0,
        },
        "review_revise_thresholds": {
            "min_quality_score": 6.5,
            "min_delta_vs_baseline": -0.50,
            "max_wrong_language_rate": 0.10,
            "max_invalid_output_rate": 0.08,
            "max_format_invalid_rate": 0.10,
            "max_failure_rate": 0.10,
            "max_latency_p95_seconds": 15.0,
        }
    },

    "output_dir": "/kaggle/working/task2_readiness"
}

OUT = Path(CONFIG["output_dir"])
for sub in [
    "logs",
    "manifests",
    "golden_eval",
    "candidate_eval",
    "outputs/baseline",
    "outputs/candidate",
    "rollback_evidence",
    "reports",
    "submission"
]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2))

In [ ]:
# Cell 4 — Helpers

LOG_PATH = OUT / "logs" / f"{RUN_ID}.log"

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def log(msg):
    line = f"[{now_iso()}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_jsonl(path, limit=None):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def percentile(values, p):
    if not values:
        return None
    vals = sorted(values)
    k = (len(vals) - 1) * p
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return vals[int(k)]
    return vals[f] * (c - k) + vals[c] * (k - f)

random.seed(CONFIG["seed"])
log(f"Run started: {RUN_ID}")

In [ ]:
# Cell 5 — Sprint 50 gold set utilities

RUBRIC_DIMENSIONS = [
    "semantic_accuracy",
    "fluency",
    "tone_register",
    "emotional_consistency",
    "cultural_appropriateness",
    "dialect_correctness",
]

REQUIRED_GOLD_COLUMNS = (
    ["sample_id", "direction", "source_en", "mt_output", "reference", "reviewer_ids"]
    + [f"{d}_gold" for d in RUBRIC_DIMENSIONS]
    + ["overall_gold", "notes"]
)

@dataclass
class GoldSample:
    sample_id: str
    direction: str
    source_en: str
    mt_output: str
    reference: Optional[str]
    reviewer_ids: str
    dim_scores: Dict[str, float]
    overall_gold: float
    notes: Optional[str] = None

def load_gold_set(csv_path: str) -> List[GoldSample]:
    df = pd.read_csv(csv_path)
    missing = [c for c in REQUIRED_GOLD_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Gold set CSV missing required columns: {missing}")

    samples = []
    for _, row in df.iterrows():
        dim_scores = {d: float(row[f"{d}_gold"]) for d in RUBRIC_DIMENSIONS}
        samples.append(
            GoldSample(
                sample_id=str(row["sample_id"]),
                direction=str(row["direction"]).upper(),
                source_en=str(row["source_en"]),
                mt_output=str(row["mt_output"]),
                reference=(str(row["reference"]) if pd.notna(row.get("reference")) else None),
                reviewer_ids=str(row["reviewer_ids"]),
                dim_scores=dim_scores,
                overall_gold=float(row["overall_gold"]),
                notes=(str(row["notes"]) if pd.notna(row.get("notes")) else None),
            )
        )
    return samples

def validate_gold_set(samples: List[GoldSample], min_per_direction: int = 30, max_per_direction: int = 50) -> Dict:
    issues = []
    counts: Dict[str, int] = {}
    seen_ids = set()

    for s in samples:
        counts[s.direction] = counts.get(s.direction, 0) + 1
        if s.sample_id in seen_ids:
            issues.append(f"Duplicate sample_id: {s.sample_id}")
        seen_ids.add(s.sample_id)

        if not (1 <= s.overall_gold <= 10):
            issues.append(f"{s.sample_id}: overall_gold out of range")
        for dim, val in s.dim_scores.items():
            if not (1 <= val <= 10):
                issues.append(f"{s.sample_id}: {dim} out of range")

    for direction, n in counts.items():
        if not (min_per_direction <= n <= int(max_per_direction * 1.2)):
            issues.append(f"Direction {direction} has {n} samples")

    return {"counts": counts, "issues": issues, "n_total": len(samples)}

In [ ]:
# Cell 6 — Sprint 50 aggregation + agreement utilities

SCORE_MIN, SCORE_MAX = 1.0, 10.0
ZERO_OUTLIER_STD_THRESHOLD = 2.0
HIGH_DISAGREEMENT_STD = 1.5
MIN_VALID_RUNS_FOR_SCORE = 2
MALFORMED_RUN_FAILURE_FRACTION = 0.5
GEMBA_LAMBDA = 0.5

@dataclass
class RunResult:
    score: Optional[float]
    parse_ok: bool
    raw_response: Optional[str] = None

def _is_out_of_range(score: float) -> bool:
    return not (SCORE_MIN <= score <= SCORE_MAX)

def aggregate_runs(runs: List[RunResult]) -> Dict[str, Any]:
    valid_scores = []
    malformed_count = 0
    n_total = len(runs)

    for r in runs:
        if not r.parse_ok or r.score is None or _is_out_of_range(r.score):
            malformed_count += 1
        else:
            valid_scores.append(r.score)

    malformed_fraction = malformed_count / n_total if n_total else 1.0

    if malformed_fraction >= MALFORMED_RUN_FAILURE_FRACTION or len(valid_scores) < MIN_VALID_RUNS_FOR_SCORE:
        return {
            "final_score": None,
            "confidence": 0.0,
            "flag": "judge_failed",
            "aggregation_method": None,
            "n_total_runs": n_total,
            "n_valid_runs": len(valid_scores),
            "malformed_fraction": round(malformed_fraction, 3),
            "flagged_outlier_scores": [],
            "valid_scores": valid_scores,
        }

    median = statistics.median(valid_scores)
    std = statistics.pstdev(valid_scores) if len(valid_scores) > 1 else 0.0

    clean_scores, flagged_outliers = [], []
    for s in valid_scores:
        is_low_outlier = std > 0 and (median - s) > ZERO_OUTLIER_STD_THRESHOLD * std
        if s <= 1.0 and is_low_outlier:
            flagged_outliers.append(s)
        else:
            clean_scores.append(s)

    scores_for_agg = clean_scores if clean_scores else valid_scores
    median2 = statistics.median(scores_for_agg)
    std2 = statistics.pstdev(scores_for_agg) if len(scores_for_agg) > 1 else 0.0
    stability = max(0.0, 1 - std2 / 5.0)
    high_disagreement = std2 >= HIGH_DISAGREEMENT_STD

    if high_disagreement:
        final_score = median2
        agg_method = "median_fallback_high_disagreement"
    else:
        weights = [1.0 / (1.0 + GEMBA_LAMBDA * abs(s - median2)) for s in scores_for_agg]
        wsum = sum(weights)
        final_score = sum(w * s for w, s in zip(weights, scores_for_agg)) / wsum
        agg_method = "gemba_weighted_mean"

    flag = "ok"
    if flagged_outliers:
        flag = "zero_outliers_excluded"
    if high_disagreement:
        flag = "high_disagreement" if flag == "ok" else flag + "+high_disagreement"

    return {
        "final_score": round(final_score, 3),
        "confidence": round(stability, 3),
        "flag": flag,
        "aggregation_method": agg_method,
        "n_total_runs": n_total,
        "n_valid_runs": len(valid_scores),
        "n_used_in_final": len(scores_for_agg),
        "malformed_fraction": round(malformed_fraction, 3),
        "flagged_outlier_scores": flagged_outliers,
        "median": round(median2, 3),
        "std": round(std2, 3),
        "valid_scores": valid_scores,
    }

def _weighted_kappa(gold: np.ndarray, pred: np.ndarray, n_buckets: int = 10) -> float:
    g = np.clip(np.round(gold), 1, n_buckets).astype(int)
    p = np.clip(np.round(pred), 1, n_buckets).astype(int)
    return float(cohen_kappa_score(g, p, weights="quadratic"))

def compute_agreement(gold: List[float], judged: List[float]) -> Dict[str, float]:
    gold_arr = np.array(gold, dtype=float)
    judged_arr = np.array(judged, dtype=float)

    if len(gold_arr) < 2:
        return {
            "pearson_r": float("nan"),
            "spearman_rho": float("nan"),
            "mae": float("nan"),
            "weighted_kappa": float("nan"),
            "n": len(gold_arr),
        }

    return {
        "pearson_r": round(float(pearsonr(gold_arr, judged_arr)[0]), 3),
        "spearman_rho": round(float(spearmanr(gold_arr, judged_arr)[0]), 3),
        "mae": round(float(np.mean(np.abs(gold_arr - judged_arr))), 3),
        "weighted_kappa": round(_weighted_kappa(gold_arr, judged_arr), 3),
        "n": len(gold_arr),
    }

In [ ]:
# Cell 7 — Hardware probe and judge selection

def hardware_probe():
    probe = {
        "timestamp_utc": now_iso(),
        "cuda_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "devices": []
    }
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            probe["devices"].append({
                "index": i,
                "name": props.name,
                "total_memory_gb": round(props.total_memory / (1024**3), 2)
            })
    return probe

def select_judge_model(hw, judge_cfg):
    max_mem = max([d["total_memory_gb"] for d in hw["devices"]], default=0)
    if max_mem >= 14:
        return judge_cfg["primary_model"], "primary", f"Selected primary judge; max GPU memory {max_mem} GB."
    elif max_mem > 0:
        return judge_cfg["fallback_model"], "fallback", f"Selected fallback judge; max GPU memory {max_mem} GB."
    return judge_cfg["fallback_model"], "fallback_cpu", "No GPU detected."

HW = hardware_probe()
JUDGE_MODEL_NAME, JUDGE_TIER, JUDGE_REASON = select_judge_model(HW, CONFIG["judge"])

save_json(HW, OUT / "manifests" / "hardware_probe_task2.json")
log(f"Judge selected: {JUDGE_MODEL_NAME} | {JUDGE_TIER} | {JUDGE_REASON}")

In [ ]:
# Cell 8 — Load Task 1 manifests and record sealed-test integrity

data_manifest = load_json(CONFIG["data"]["data_manifest_path"])
candidate_manifest = load_json(CONFIG["candidate"]["candidate_manifest_path"])

sealed_integrity = {
    "sealed_final_test_path_configured": CONFIG["data"]["sealed_final_test_path"],
    "sealed_final_test_path_from_manifest": data_manifest["files"]["sealed_final_test"]["path"],
    "sealed_final_test_sha256_from_manifest": data_manifest["files"]["sealed_final_test"]["sha256"],
    "opened_for_eval": False,
    "note": "Task 2 does not open sealed final test contents; uses manifest hash only."
}

save_json(sealed_integrity, OUT / "manifests" / "sealed_test_integrity.json")
log("Sealed test integrity record saved")

In [ ]:
# Cell 9 — Load Task 1 eval data and filter to en->zh

def normalize_task1_row(row):
    return {
        "id": row["id"],
        "pair_id": row["pair_id"],
        "source_text": row["source_text"],
        "target_text": row["target_text"],
        "source_lang": row["source_lang"],
        "target_lang": row["target_lang"],
        "domain": row.get("domain", "unknown"),
        "license": row.get("license", ""),
        "dataset_name": row.get("dataset_name", "")
    }

prompt_validation_rows = [normalize_task1_row(r) for r in load_jsonl(CONFIG["data"]["prompt_validation_path"])]
prompt_validation_en2zh = [
    r for r in prompt_validation_rows
    if r["source_lang"] == "en" and r["target_lang"] == "zh"
]

n_eval = CONFIG["dry_run_n"] if CONFIG["dry_run"] else min(CONFIG["data"]["eval_sample_size"], len(prompt_validation_en2zh))
eval_examples = random.sample(prompt_validation_en2zh, n_eval) if len(prompt_validation_en2zh) >= n_eval else prompt_validation_en2zh

save_json({
    "total_prompt_validation_rows": len(prompt_validation_rows),
    "filtered_en2zh_rows": len(prompt_validation_en2zh),
    "selected_eval_rows": len(eval_examples),
    "dry_run": CONFIG["dry_run"]
}, OUT / "manifests" / "evaluation_dataset_selection.json")

log(f"Selected {len(eval_examples)} en->zh eval rows")

In [ ]:
# Cell 10 — Generation prompts

CANDIDATE_SYSTEM_PROMPT = (
    "You are an expert translator for English to Mandarin Chinese (普通話). "
    "Rules you must follow:\n"
    "1. Output ONLY the Mandarin translation in Simplified Chinese characters.\n"
    "2. Use standard Mainland China Mandarin vocabulary and grammar.\n"
    "3. Do NOT use Cantonese vocabulary or particles.\n"
    "4. Do NOT romanize and do not use Pinyin.\n"
    "5. Do NOT explain, repeat the source, or add commentary.\n"
    "6. If a proper noun has no Mandarin equivalent, keep the original English term."
)

BASELINE_RECONSTRUCT_PROMPT = (
    "You are a professional English-to-Mandarin Chinese translator. "
    "Translate the user's sentence into natural, fluent Simplified Chinese, preserving tone, intent, "
    "and idiomatic meaning. Output only the Chinese translation with no romanization, explanation, or commentary."
)

In [ ]:
# Cell 11 — Candidate/baseline model loaders

def load_unsloth_base_and_tokenizer(model_name, max_seq_length=1024, force_chatml=False):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name, max_seq_length=max_seq_length, dtype=None, load_in_4bit=True,
    )
    if force_chatml or getattr(tokenizer, "chat_template", None) is None:
        tokenizer = get_chat_template(tokenizer, chat_template="chatml")
    # else: use the model's own native template as-is (this is the Instruct-model case)
    FastLanguageModel.for_inference(model)
    return model, tokenizer


def load_candidate_system():
    ctype = CONFIG["candidate"]["candidate_type"]
    if ctype == "lora_adapter":
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=CONFIG["candidate"]["adapter_path"],   # final_dir, not base_model
            max_seq_length=1024,
            dtype=None,
            load_in_4bit=True,
        )
        FastLanguageModel.for_inference(model)
        return model, tokenizer, "candidate", CONFIG["candidate"]["version"]
    elif ctype == "merged_model":
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=CONFIG["candidate"]["merged_model_path"],
            max_seq_length=1024,
            dtype=None,
            load_in_4bit=True,
        )
        FastLanguageModel.for_inference(model)
        return model, tokenizer, "candidate", CONFIG["candidate"]["version"]

    else:
        raise ValueError(f"Unknown candidate_type: {ctype}")

def generate_with_chat(model, tokenizer, system_prompt, src_text, max_new_tokens=256):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Translate to Mandarin:\n{src_text}"}
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            use_cache=True
        )

    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def build_baseline_generator():
    mode = CONFIG["baseline"]["mode"]

    if mode == "fixture":
        fixture_rows = load_jsonl(CONFIG["baseline"]["fixture_path"])
        lookup = {r["source_text"]: r["translation"] for r in fixture_rows}

        def fn(src):
            return lookup.get(src, "")

        manifest = {
            "baseline_mode": "fixture",
            "baseline_version": CONFIG["baseline"]["version"],
            "fixture_path": CONFIG["baseline"]["fixture_path"]
        }
        return fn, manifest, None, None

    elif mode == "reconstruct":
        model, tokenizer = load_unsloth_base_and_tokenizer(CONFIG["baseline"]["reconstruct_base_model"])

        def fn(src):
            return generate_with_chat(model, tokenizer, BASELINE_RECONSTRUCT_PROMPT, src)

        manifest = {
            "baseline_mode": "reconstruct",
            "baseline_version": CONFIG["baseline"]["version"],
            "base_model": CONFIG["baseline"]["reconstruct_base_model"],
            "prompt_sha256": sha256_text(BASELINE_RECONSTRUCT_PROMPT)
        }
        return fn, manifest, model, tokenizer

    else:
        raise ValueError("Unsupported baseline mode")

In [ ]:
# Cell 12 — Run generation for one system

def run_generation(generate_fn, examples, system_label, system_version):
    rows = []
    failures = 0

    for ex in examples:
        t0 = time.time()
        rec = {
            "id": ex["id"],
            "pair_id": ex["pair_id"],
            "source_text": ex["source_text"],
            "reference_text": ex["target_text"],
            "source_lang": ex["source_lang"],
            "target_lang": ex["target_lang"],
            "domain": ex["domain"],
            "system": system_label,
            "system_version": system_version
        }

        try:
            rec["output_text"] = generate_fn(ex["source_text"])
            rec["status"] = "ok"
        except Exception as e:
            rec["output_text"] = ""
            rec["status"] = "failure"
            rec["error"] = repr(e)
            failures += 1

        rec["latency_seconds"] = round(time.time() - t0, 4)
        rows.append(rec)

    log(f"{system_label} done: {len(rows)} rows, failures={failures}")
    return rows

In [ ]:
# Cell 13 — Generate baseline outputs

baseline_generate_fn, baseline_manifest, baseline_model, baseline_tokenizer = build_baseline_generator()
save_json(baseline_manifest, OUT / "manifests" / "baseline_manifest.json")

baseline_outputs = run_generation(
    baseline_generate_fn,
    eval_examples,
    "baseline",
    CONFIG["baseline"]["version"]
)

baseline_outputs_path = OUT / "outputs" / "baseline" / f"{RUN_ID}_baseline_outputs.jsonl"
save_jsonl(baseline_outputs, baseline_outputs_path)

if baseline_model is not None:
    del baseline_model
if baseline_tokenizer is not None:
    del baseline_tokenizer
clear_memory()

log(f"Baseline outputs saved: {baseline_outputs_path}")

In [ ]:
# Cell 14 — Generate candidate outputs
candidate_model, candidate_tokenizer, candidate_label, candidate_version = load_candidate_system()

def candidate_generate_fn(src):
    return generate_with_chat(candidate_model, candidate_tokenizer, CANDIDATE_SYSTEM_PROMPT, src)

candidate_outputs = run_generation(
    candidate_generate_fn,
    eval_examples,
    candidate_label,
    candidate_version
)

candidate_outputs_path = OUT / "outputs" / "candidate" / f"{RUN_ID}_candidate_outputs.jsonl"
save_jsonl(candidate_outputs, candidate_outputs_path)

del candidate_model
del candidate_tokenizer
clear_memory()

log(f"Candidate outputs saved: {candidate_outputs_path}")

In [ ]:
# Cell 15 — Sprint 50-style LocalJudgeModel for Task 2

class LocalJudgeModel:
    def __init__(
        self,
        model_name: str,
        max_seq_length: int = 4096,
        max_new_tokens: int = 256,
        temperature: float = 0.0,
        load_in_4bit: bool = True,
    ):
        self.model_name = model_name
        self.max_seq_length = max_seq_length
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            dtype=None,
            load_in_4bit=load_in_4bit,
        )
        FastLanguageModel.for_inference(self.model)

    def build_judge_prompt(self, english: str, mandarin: str) -> str:
        return (
            "<|im_start|>system\n"
            "You are a strict professional translation evaluator.\n"
            "Evaluate the Mandarin translation for accuracy, fluency, and fidelity.\n"
            "Return ONLY valid JSON in this exact format:\n"
            '{"score": <number from 1 to 10>, "feedback": "<short explanation>"}\n'
            "Do not add any extra text.\n"
            "Do not think aloud.\n"
            "<|im_end|>\n"
            "<|im_start|>user\n"
            f"English source:\n{english}\n\n"
            f"Mandarin translation:\n{mandarin}\n"
            "<|im_end|>\n"
            "<|im_start|>assistant\n"
            "/no_think\n"
        )

    def clean_output(self, text: str) -> str:
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
        text = re.sub(r"</?think>", "", text)
        return text.strip()

    def evaluate(self, english: str, mandarin: str = "") -> Dict:
        try:
            prompt = self.build_judge_prompt(english, mandarin)
            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=self.max_seq_length,
            ).to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=self.temperature > 0,
                    temperature=self.temperature if self.temperature > 0 else None,
                    renormalize_logits=True,
                    use_cache=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )

            generated_tokens = outputs[:, inputs.input_ids.shape[1]:]
            raw_output = self.tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
            raw_output = self.clean_output(raw_output)

            score = None
            feedback = raw_output
            try:
                parsed = json.loads(raw_output)
                score = float(parsed.get("score"))
                feedback = parsed.get("feedback", raw_output)
            except Exception:
                match = re.search(r'(\d+(?:\.\d+)?)', raw_output)
                if match:
                    num = float(match.group(1))
                    if 1 <= num <= 10:
                        score = num

            return {
                "score": round(score, 2) if score is not None else None,
                "feedback": feedback,
                "success": score is not None,
                "error": None if score is not None else f"Could not parse judge output: {raw_output}",
                "raw_output": raw_output
            }
        except Exception as e:
            return {
                "score": None,
                "feedback": None,
                "success": False,
                "error": str(e),
                "raw_output": None
            }

In [ ]:
# Cell 16 — Load gold set and calibrate judge on EN_CMN only

gold_samples_all = load_gold_set(CONFIG["data"]["gold_set_csv_path"])
gold_validation = validate_gold_set(gold_samples_all)

save_json(gold_validation, OUT / "golden_eval" / f"{RUN_ID}_gold_set_validation.json")
log(f"Gold set validation: {gold_validation}")

gold_samples_cmn = [s for s in gold_samples_all if s.direction == "EN_CMN"]
if CONFIG["dry_run"]:
    gold_samples_cmn = gold_samples_cmn[:CONFIG["dry_run_n"]]

log(f"Mandarin golden samples for calibration: {len(gold_samples_cmn)}")

judge_model = LocalJudgeModel(
    model_name=JUDGE_MODEL_NAME,
    max_seq_length=CONFIG["judge"]["max_seq_length"],
    max_new_tokens=CONFIG["judge"]["max_new_tokens"],
    temperature=CONFIG["judge"]["temperature"],
    load_in_4bit=CONFIG["judge"]["load_in_4bit"],
)

In [ ]:
# Cell 17 — Judge calibration scoring

golden_rows = []
for sample in gold_samples_cmn:
    raw_runs = []
    for _ in range(CONFIG["judge"]["num_runs"]):
        result = judge_model.evaluate(english=sample.source_en, mandarin=sample.mt_output)
        if result.get("success") and result.get("score") is not None:
            raw_runs.append(RunResult(score=float(result["score"]), parse_ok=True, raw_response=str(result)))
        else:
            raw_runs.append(RunResult(score=None, parse_ok=False, raw_response=str(result)))

    agg = aggregate_runs(raw_runs)
    golden_rows.append({
        "sample_id": sample.sample_id,
        "direction": sample.direction,
        "gold_overall": sample.overall_gold,
        "judge_final_score": agg["final_score"],
        "judge_confidence": agg["confidence"],
        "judge_flag": agg["flag"],
        "n_valid_runs": agg["n_valid_runs"],
        "n_total_runs": agg["n_total_runs"],
        "aggregation_method": agg.get("aggregation_method"),
        "malformed_fraction": agg.get("malformed_fraction"),
        "flagged_outlier_scores": agg.get("flagged_outlier_scores", []),
        "valid_scores": agg.get("valid_scores", []),
    })

golden_scored_df = pd.DataFrame(golden_rows)
golden_scored_path = OUT / "golden_eval" / f"{RUN_ID}_golden_scored_en_cmn.csv"
golden_scored_df.to_csv(golden_scored_path, index=False)

usable = golden_scored_df[golden_scored_df["judge_flag"] != "judge_failed"]
agreement = compute_agreement(
    usable["gold_overall"].tolist(),
    usable["judge_final_score"].tolist()
)

coverage = round(len(usable) / len(golden_scored_df), 3) if len(golden_scored_df) else 0.0
mean_confidence = round(float(usable["judge_confidence"].mean()), 3) if len(usable) else 0.0

judge_calibration_summary = {
    "judge_model": JUDGE_MODEL_NAME,
    "judge_tier": JUDGE_TIER,
    "selection_reason": JUDGE_REASON,
    "direction": "EN_CMN",
    "n_total_samples": int(len(golden_scored_df)),
    "n_judge_failed": int(len(golden_scored_df) - len(usable)),
    "coverage": coverage,
    "mean_confidence": mean_confidence,
    **agreement,
    "thresholds": CONFIG["judge"]["calibration_thresholds"],
    "scored_file": str(golden_scored_path),
}

save_json(judge_calibration_summary, OUT / "golden_eval" / f"{RUN_ID}_golden_summary_en_cmn.json")
log(f"Judge calibration summary: {judge_calibration_summary}")

th = CONFIG["judge"]["calibration_thresholds"]
judge_calibration_pass = (
    coverage >= th["min_coverage"] and
    (not pd.isna(judge_calibration_summary["spearman_rho"])) and
    judge_calibration_summary["spearman_rho"] >= th["min_spearman_rho"] and
    mean_confidence >= th["min_mean_confidence"]
)

if not judge_calibration_pass:
    raise RuntimeError("Judge calibration failed; aborting before candidate scoring.")

In [ ]:
# Cell 18 — Freeze judge_registry.json

judge_registry = {
    "registry_version": "judge-registry-v1",
    "frozen_at": now_iso(),
    "run_id": RUN_ID,
    "judge_model": JUDGE_MODEL_NAME,
    "judge_tier": JUDGE_TIER,
    "selection_reason": JUDGE_REASON,
    "max_seq_length": CONFIG["judge"]["max_seq_length"],
    "max_new_tokens": CONFIG["judge"]["max_new_tokens"],
    "temperature": CONFIG["judge"]["temperature"],
    "load_in_4bit": CONFIG["judge"]["load_in_4bit"],
    "num_runs": CONFIG["judge"]["num_runs"],
    "judge_prompt_type": "sprint50_local_judge_mandarin_json_score_only",
    "golden_calibration": judge_calibration_summary,
}

judge_registry_path = OUT / "manifests" / "judge_registry.json"
save_json(judge_registry, judge_registry_path)
log(f"Judge registry frozen: {judge_registry_path}")

In [ ]:
# Cell 19 — Score baseline and candidate outputs with the frozen judge

frozen_judge = load_json(judge_registry_path)

assert frozen_judge["judge_model"] == JUDGE_MODEL_NAME
assert frozen_judge["num_runs"] == CONFIG["judge"]["num_runs"]
assert frozen_judge["temperature"] == CONFIG["judge"]["temperature"]
assert frozen_judge["max_new_tokens"] == CONFIG["judge"]["max_new_tokens"]

def score_outputs_with_sprint50_judge(rows, judge_model_obj, num_runs):
    scored = []
    for r in rows:
        if r["status"] != "ok":
            scored.append({
                **r,
                "judge_result": {
                    "final_score": None,
                    "confidence": 0.0,
                    "flag": "generation_failed"
                }
            })
            continue

        raw_runs = []
        for _ in range(num_runs):
            res = judge_model_obj.evaluate(english=r["source_text"], mandarin=r["output_text"])
            if res.get("success") and res.get("score") is not None:
                raw_runs.append(RunResult(score=float(res["score"]), parse_ok=True, raw_response=str(res)))
            else:
                raw_runs.append(RunResult(score=None, parse_ok=False, raw_response=str(res)))

        agg = aggregate_runs(raw_runs)
        scored.append({**r, "judge_result": agg})
    return scored

scored_baseline = score_outputs_with_sprint50_judge(baseline_outputs, judge_model, frozen_judge["num_runs"])
scored_candidate = score_outputs_with_sprint50_judge(candidate_outputs, judge_model, frozen_judge["num_runs"])

save_json(scored_baseline, OUT / "candidate_eval" / f"{RUN_ID}_baseline_scored.json")
save_json(scored_candidate, OUT / "candidate_eval" / f"{RUN_ID}_candidate_scored.json")
log("Baseline and candidate scored with frozen judge")

In [ ]:
# Cell 20 — Extra guardrail heuristics

CANTONESE_MARKERS = ["佢", "咗", "喺", "啲", "冇", "哋", "嚟", "嘅"]
ROLE_TAG_PATTERNS = [r"<\|im_start\|>", r"<\|im_end\|>", r"^system\s", r"^user\s", r"^assistant\s"]
PINYIN_PATTERN = r"\b(?:[a-zA-Z]+[1-5])\b"

def cjk_count(text):
    return sum(1 for ch in text if "\u4e00" <= ch <= "\u9fff")

def latin_count(text):
    return sum(1 for ch in text if ("a" <= ch.lower() <= "z"))

def has_role_tags(text):
    return any(re.search(p, text.strip(), flags=re.IGNORECASE) for p in ROLE_TAG_PATTERNS)

def contains_pinyin_like(text):
    return re.search(PINYIN_PATTERN, text) is not None

def contains_cantonese_markers(text):
    return any(tok in text for tok in CANTONESE_MARKERS)

def wrong_language_heuristic(text):
    if not text or not text.strip():
        return True
    total = max(1, len(text))
    if cjk_count(text) / total < 0.20:
        return True
    if latin_count(text) / total > 0.45:
        return True
    if contains_pinyin_like(text):
        return True
    if contains_cantonese_markers(text):
        return True
    return False

def invalid_output_heuristic(text):
    if not text or not text.strip():
        return True
    stripped = text.strip()
    if len(stripped) <= 1:
        return True
    if has_role_tags(stripped):
        return True
    if "translate to mandarin" in stripped.lower():
        return True
    if stripped.lower().startswith("here is") or stripped.lower().startswith("translation:"):
        return True
    if re.fullmatch(r"[\W_]+", stripped):
        return True
    return False

def format_invalid_heuristic(text):
    if invalid_output_heuristic(text):
        return True
    if contains_pinyin_like(text):
        return True
    if has_role_tags(text):
        return True
    return False

def normalized_text(s):
    return re.sub(r"\s+", " ", s.strip().lower()) if s else ""

train_rows_for_leak = load_jsonl(
    CONFIG["data"]["finetune_train_path"],
    limit=(3000 if not CONFIG["dry_run"] else 200)
)

train_targets_en2zh = {
    normalized_text(r["target_text"])
    for r in train_rows_for_leak
    if r["source_lang"] == "en" and r["target_lang"] == "zh"
}

def memorization_heuristic(text):
    nt = normalized_text(text)
    return nt in train_targets_en2zh if nt else False

SAFETY_PATTERNS = [
    r"自杀", r"炸弹", r"恐怖", r"杀死", r"攻击", r"武器制作",
    r"suicide", r"bomb", r"terror", r"kill", r"attack", r"weapon"
]

def safety_heuristic(text):
    if not text:
        return False
    low = text.lower()
    return any(re.search(p, low, flags=re.IGNORECASE) for p in SAFETY_PATTERNS)

In [ ]:
# Cell 21 — Aggregate final metrics

def aggregate_metrics(scored_rows, label):
    n = len(scored_rows)
    failures = sum(1 for r in scored_rows if r["status"] != "ok")
    valid_rows = [r for r in scored_rows if r["status"] == "ok"]

    quality_scores = []
    enriched = []

    wrong_lang = invalid = safety = memorization = format_invalid = 0

    for r in scored_rows:
        out = r.get("output_text", "")
        jr = r.get("judge_result", {})

        heur_wrong = wrong_language_heuristic(out)
        heur_invalid = invalid_output_heuristic(out)
        heur_format_invalid = format_invalid_heuristic(out)
        heur_mem = memorization_heuristic(out)
        heur_safe = safety_heuristic(out)

        judge_failed = jr.get("flag") == "judge_failed"
        judge_score = jr.get("final_score")
        judge_conf = jr.get("confidence", 0.0)

        if r["status"] == "ok" and not judge_failed and judge_score is not None:
            quality_scores.append(judge_score)

        final_wrong = heur_wrong
        final_invalid = heur_invalid
        final_mem = heur_mem
        final_safety = heur_safe
        final_format_invalid = heur_format_invalid

        if r["status"] == "ok":
            wrong_lang += int(final_wrong)
            invalid += int(final_invalid)
            memorization += int(final_mem)
            safety += int(final_safety)
            format_invalid += int(final_format_invalid)

        enriched.append({
            **r,
            "checks": {
                "wrong_language_final": final_wrong,
                "invalid_output_final": final_invalid,
                "memorization_final": final_mem,
                "safety_final": final_safety,
                "format_invalid_final": final_format_invalid,
                "judge_failed": judge_failed,
                "judge_confidence": judge_conf,
            }
        })

    latencies = [r["latency_seconds"] for r in scored_rows]
    metrics = {
        "system": label,
        "n_examples": n,
        "n_failures": failures,
        "failure_rate": failures / n if n else None,
        "mean_quality_score": (sum(quality_scores) / len(quality_scores)) if quality_scores else None,
        "n_scored": len(quality_scores),
        "wrong_language_rate": wrong_lang / max(1, len(valid_rows)),
        "invalid_output_rate": invalid / max(1, len(valid_rows)),
        "safety_violation_rate": safety / max(1, len(valid_rows)),
        "memorization_flag_rate": memorization / max(1, len(valid_rows)),
        "format_invalid_rate": format_invalid / max(1, len(valid_rows)),
        "latency_p50_seconds": percentile(latencies, 0.50),
        "latency_p95_seconds": percentile(latencies, 0.95),
        "mean_judge_confidence": round(float(np.mean([
            r["judge_result"]["confidence"] for r in scored_rows
            if r.get("judge_result", {}).get("confidence") is not None
        ])), 3) if scored_rows else None,
    }
    return metrics, enriched

baseline_metrics, baseline_enriched = aggregate_metrics(scored_baseline, "baseline")
candidate_metrics, candidate_enriched = aggregate_metrics(scored_candidate, "candidate")

save_json(baseline_enriched, OUT / "candidate_eval" / f"{RUN_ID}_baseline_enriched.json")
save_json(candidate_enriched, OUT / "candidate_eval" / f"{RUN_ID}_candidate_enriched.json")

quality_delta = None
if baseline_metrics["mean_quality_score"] is not None and candidate_metrics["mean_quality_score"] is not None:
    quality_delta = candidate_metrics["mean_quality_score"] - baseline_metrics["mean_quality_score"]

domain_regression = {
    "quality_delta_vs_baseline": quality_delta,
    "regressed": quality_delta is not None and quality_delta < 0
}

quality_summary = {
    "baseline_metrics": baseline_metrics,
    "candidate_metrics": candidate_metrics,
    "domain_regression": domain_regression
}
save_json(quality_summary, OUT / "candidate_eval" / f"{RUN_ID}_quality_summary.json")
log(f"Quality delta vs baseline: {quality_delta}")

In [ ]:
# Cell 22 — Freeze readiness_policy.json

readiness_policy = {
    "policy_version": CONFIG["policy"]["policy_version"],
    "frozen_at": now_iso(),
    "advance_thresholds": CONFIG["policy"]["advance_thresholds"],
    "review_revise_thresholds": CONFIG["policy"]["review_revise_thresholds"],
    "critical_reject_conditions": [
        "safety_violation_rate > 0",
        "memorization_flag_rate > 0",
        "sealed_test_contamination_detected == true",
        "rollback_smoke_test_failed == true",
    ]
}

readiness_policy_path = OUT / "manifests" / "readiness_policy.json"
save_json(readiness_policy, readiness_policy_path)
log(f"Readiness policy frozen: {readiness_policy_path}")

In [ ]:
# Cell 23 — Rollback evidence

rollback_generate_fn, rollback_manifest, rollback_model, rollback_tokenizer = build_baseline_generator()

rollback_input = eval_examples[0]["source_text"] if eval_examples else "The weather is nice today."
rollback_failed = False
rollback_output = None

try:
    rollback_output = rollback_generate_fn(rollback_input)
    if not rollback_output or not rollback_output.strip():
        rollback_failed = True
except Exception as e:
    rollback_output = f"ERROR: {repr(e)}"
    rollback_failed = True

if rollback_model is not None:
    del rollback_model
if rollback_tokenizer is not None:
    del rollback_tokenizer
clear_memory()

leakage_flagged_examples = [
    {"id": r["id"], "output_text": r["output_text"]}
    for r in candidate_enriched
    if r["checks"]["memorization_final"]
]

rollback_evidence = {
    "run_id": RUN_ID,
    "timestamp_utc": now_iso(),
    "baseline_version": CONFIG["baseline"]["version"],
    "baseline_mode": CONFIG["baseline"]["mode"],
    "smoke_test_input": rollback_input,
    "smoke_test_output": rollback_output,
    "rollback_smoke_test_failed": rollback_failed,
    "sealed_test_integrity": sealed_integrity,
    "sealed_test_contamination_detected": False,
    "memorization_leakage_scan": {
        "train_sample_size": len(train_targets_en2zh),
        "n_flagged": len(leakage_flagged_examples),
        "flagged_examples_preview": leakage_flagged_examples[:20]
    }
}

rollback_evidence_path = OUT / "rollback_evidence" / f"{RUN_ID}_rollback_evidence.json"
save_json(rollback_evidence, rollback_evidence_path)
log(f"Rollback evidence saved: {rollback_evidence_path}")

In [ ]:
# Cell 24 — Decision engine

def decide(candidate_metrics, baseline_metrics, domain_regression, rollback_evidence, policy):
    reasons = []

    critical = (
        (candidate_metrics["safety_violation_rate"] or 0) > 0 or
        (candidate_metrics["memorization_flag_rate"] or 0) > 0 or
        rollback_evidence["sealed_test_contamination_detected"] or
        rollback_evidence["rollback_smoke_test_failed"]
    )
    if critical:
        if (candidate_metrics["safety_violation_rate"] or 0) > 0:
            reasons.append("critical reject: safety violation")
        if (candidate_metrics["memorization_flag_rate"] or 0) > 0:
            reasons.append("critical reject: memorization or leakage")
        if rollback_evidence["sealed_test_contamination_detected"]:
            reasons.append("critical reject: sealed test contamination")
        if rollback_evidence["rollback_smoke_test_failed"]:
            reasons.append("critical reject: rollback smoke test failed")
        return "reject", reasons

    adv = policy["advance_thresholds"]
    delta = domain_regression["quality_delta_vs_baseline"]

    advance_ok = (
        (candidate_metrics["mean_quality_score"] or 0) >= adv["min_quality_score"] and
        (delta if delta is not None else -999) >= adv["min_delta_vs_baseline"] and
        (candidate_metrics["wrong_language_rate"] or 0) <= adv["max_wrong_language_rate"] and
        (candidate_metrics["invalid_output_rate"] or 0) <= adv["max_invalid_output_rate"] and
        (candidate_metrics["safety_violation_rate"] or 0) <= adv["max_safety_violation_rate"] and
        (candidate_metrics["memorization_flag_rate"] or 0) <= adv["max_memorization_flag_rate"] and
        (candidate_metrics["format_invalid_rate"] or 0) <= adv["max_format_invalid_rate"] and
        (candidate_metrics["failure_rate"] or 0) <= adv["max_failure_rate"] and
        (candidate_metrics["latency_p95_seconds"] or 999) <= adv["max_latency_p95_seconds"]
    )
    if advance_ok:
        return "advance", ["all advance thresholds met"]

    rev = policy["review_revise_thresholds"]
    review_ok = (
        (candidate_metrics["mean_quality_score"] or 0) >= rev["min_quality_score"] and
        (delta if delta is not None else -999) >= rev["min_delta_vs_baseline"] and
        (candidate_metrics["wrong_language_rate"] or 0) <= rev["max_wrong_language_rate"] and
        (candidate_metrics["invalid_output_rate"] or 0) <= rev["max_invalid_output_rate"] and
        (candidate_metrics["format_invalid_rate"] or 0) <= rev["max_format_invalid_rate"] and
        (candidate_metrics["failure_rate"] or 0) <= rev["max_failure_rate"] and
        (candidate_metrics["latency_p95_seconds"] or 999) <= rev["max_latency_p95_seconds"]
    )
    if review_ok:
        return "review-revise", ["met review-revise thresholds but not advance thresholds"]

    return "reject", ["failed review-revise thresholds; reject and keep baseline"]

DECISION, DECISION_REASONS = decide(
    candidate_metrics,
    baseline_metrics,
    domain_regression,
    rollback_evidence,
    readiness_policy
)

log(f"FINAL DECISION: {DECISION}")
log(f"Reasons: {DECISION_REASONS}")

In [ ]:
# Cell 25 — Save summary and reports

evaluation_summary = {
    "run_id": RUN_ID,
    "generated_at": now_iso(),
    "candidate_version": CONFIG["candidate"]["version"],
    "candidate_manifest_ref": CONFIG["candidate"]["candidate_manifest_path"],
    "baseline_version": CONFIG["baseline"]["version"],
    "baseline_mode": CONFIG["baseline"]["mode"],
    "judge_registry_ref": str(judge_registry_path),
    "readiness_policy_ref": str(readiness_policy_path),
    "decision": DECISION,
    "decision_reasons": DECISION_REASONS,
    "baseline_metrics": baseline_metrics,
    "candidate_metrics": candidate_metrics,
    "domain_regression": domain_regression,
    "rollback_evidence_ref": str(rollback_evidence_path)
}
evaluation_summary_path = OUT / "candidate_eval" / f"{RUN_ID}_evaluation_summary.json"
save_json(evaluation_summary, evaluation_summary_path)

readiness_report_json = {
    "run_id": RUN_ID,
    "generated_at": now_iso(),
    "decision": DECISION,
    "decision_reasons": DECISION_REASONS,
    "candidate": {
        "version": CONFIG["candidate"]["version"],
        "type": CONFIG["candidate"]["candidate_type"],
        "base_model": CONFIG["candidate"]["base_model"],
        "candidate_manifest_path": CONFIG["candidate"]["candidate_manifest_path"]
    },
    "baseline": {
        "version": CONFIG["baseline"]["version"],
        "mode": CONFIG["baseline"]["mode"]
    },
    "judge": {
        "model_used": JUDGE_MODEL_NAME,
        "tier": JUDGE_TIER,
        "judge_registry_path": str(judge_registry_path),
        "golden_calibration": judge_calibration_summary
    },
    "metrics": {
        "baseline": baseline_metrics,
        "candidate": candidate_metrics,
        "domain_regression": domain_regression
    },
    "rollback": rollback_evidence,
    "hardware_probe": HW
}
report_json_path = OUT / "reports" / f"{RUN_ID}_readiness_report.json"
save_json(readiness_report_json, report_json_path)

report_md_path = OUT / "reports" / f"{RUN_ID}_readiness_report.md"
with open(report_md_path, "w", encoding="utf-8") as f:
    f.write(f"""# Sprint 51 Task 2 Readiness Report

## Decision
**{DECISION.upper()}**

## Reasons
- """ + "\n- ".join(DECISION_REASONS) + f"""

## Candidate
- Version: {CONFIG['candidate']['version']}
- Type: {CONFIG['candidate']['candidate_type']}
- Base model: {CONFIG['candidate']['base_model']}
- Manifest: {CONFIG['candidate']['candidate_manifest_path']}

## Baseline
- Version: {CONFIG['baseline']['version']}
- Mode: {CONFIG['baseline']['mode']}

## Judge
- Model: {JUDGE_MODEL_NAME}
- Tier: {JUDGE_TIER}
- Golden direction: EN_CMN
- Coverage: {judge_calibration_summary['coverage']}
- Spearman rho: {judge_calibration_summary['spearman_rho']}
- Pearson r: {judge_calibration_summary['pearson_r']}
- MAE: {judge_calibration_summary['mae']}
- Weighted kappa: {judge_calibration_summary['weighted_kappa']}
- Mean confidence: {judge_calibration_summary['mean_confidence']}

## Candidate vs Baseline
| Metric | Baseline | Candidate |
|---|---:|---:|
| Mean quality score | {baseline_metrics['mean_quality_score']} | {candidate_metrics['mean_quality_score']} |
| Wrong-language rate | {baseline_metrics['wrong_language_rate']} | {candidate_metrics['wrong_language_rate']} |
| Invalid-output rate | {baseline_metrics['invalid_output_rate']} | {candidate_metrics['invalid_output_rate']} |
| Safety violation rate | {baseline_metrics['safety_violation_rate']} | {candidate_metrics['safety_violation_rate']} |
| Memorization flag rate | {baseline_metrics['memorization_flag_rate']} | {candidate_metrics['memorization_flag_rate']} |
| Format-invalid rate | {baseline_metrics['format_invalid_rate']} | {candidate_metrics['format_invalid_rate']} |
| Failure rate | {baseline_metrics['failure_rate']} | {candidate_metrics['failure_rate']} |
| Latency p95 (s) | {baseline_metrics['latency_p95_seconds']} | {candidate_metrics['latency_p95_seconds']} |

## Domain regression
- Quality delta vs baseline: {domain_regression['quality_delta_vs_baseline']}
- Regressed: {domain_regression['regressed']}

## Rollback
- Baseline smoke test failed: {rollback_evidence['rollback_smoke_test_failed']}
- Sealed test contamination detected: {rollback_evidence['sealed_test_contamination_detected']}
- Leakage scan flagged outputs: {rollback_evidence['memorization_leakage_scan']['n_flagged']}
""")

log(f"Reports saved: {report_json_path} and {report_md_path}")

In [ ]:
# Cell 26 — Package submission archive

submission_manifest = {
    "run_id": RUN_ID,
    "created_at": now_iso(),
    "artifacts": {
        "judge_registry": str(judge_registry_path),
        "readiness_policy": str(readiness_policy_path),
        "evaluation_summary": str(evaluation_summary_path),
        "readiness_report_json": str(report_json_path),
        "readiness_report_md": str(report_md_path),
        "rollback_evidence": str(rollback_evidence_path),
        "baseline_outputs": str(baseline_outputs_path),
        "candidate_outputs": str(candidate_outputs_path),
        "golden_scored_en_cmn": str(golden_scored_path),
    },
    "github_link": "TODO: replace with exact GitHub link after push"
}

save_json(submission_manifest, OUT / "submission" / "submission_manifest.json")

archive_base = f"/kaggle/working/{RUN_ID}_task2_submission"
shutil.make_archive(archive_base, "zip", OUT)

print("✓ Submission archive:", archive_base + ".zip")
print("✓ Final decision:", DECISION.upper())
print("✓ Reasons:", DECISION_REASONS)

In [ ]:
# Cell 27 — Final quick summary

print("\n================ TASK 2 COMPLETE ================\n")
print("Main outputs:")
print("-", judge_registry_path)
print("-", readiness_policy_path)
print("-", evaluation_summary_path)
print("-", report_json_path)
print("-", report_md_path)
print("-", rollback_evidence_path)
print("-", archive_base + ".zip")
print("\nFinal decision:", DECISION.upper())